In [1]:
from tqdm.auto import tqdm
import pandas as pd
import polars as pl
import numpy as np
import argparse
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from typing import Optional
from typing import Tuple
from typing import List
from typing import Dict
from typing import Any

In [2]:
DATA_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
SPEC_DIR = os.path.join(DATA_DIR, "test_spectrograms/")
EEG_DIR = os.path.join(DATA_DIR, "test_eegs/")

df_train = pl.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test = pl.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

df_test.head()

spectrogram_id,eeg_id,patient_id
i64,i64,i64
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
8352559,1706196508,37552


In [3]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cpu



In [4]:
def load_model(path: str, model: nn.Module) -> nn.Module:
    model.load_state_dict(torch.load(path, map_location = torch.device("cpu"))["model_state_dict"])
    return model

In [5]:
import sys
sys.path.append("/kaggle/input/hms-models/")

from eeg_cnn_rnn_att import EegModel as EegModel
from spc_cnn_att import SpectrogramCnnModel as SpcModel
from comb_model import MultimodalModel

ModuleNotFoundError: No module named 'eeg_cnn_rnn_att'

In [6]:
import glob
import sys

# Individual models
"""
models_eeg = [] 
for model_path in sorted(glob.glob("/kaggle/input/hms-models/kaggle_send/kaggle_send/*")):
    model = EegModel()
    print(model_path)
    model = load_model(model_path, model)
    model = model.to(device)
    model.eval()
    models_eeg.append(model) 

models_spc = []
for model_path in sorted(glob.glob("/kaggle/input/hms-models/final_spec_kaggle/final_spec_kaggle/*")):
    model = SpcModel()
    print(model_path)
    model = load_model(model_path, model)
    model = model.to(device)
    model.eval()
    models_spc.append(model)

models_eeg_spc = []
for model_path in sorted(glob.glob("/kaggle/input/hms-models/final_eeg_spec_kaggle/final_eeg_spec_kaggle/*"))[::-1]:
    model = SpcModel()
    print(model_path)
    model = load_model(model_path, model)
    model = model.to(device)
    model.eval()
    models_eeg_spc.append(model)
"""

# Ensemble model
models_multimodal = []
for model_path in sorted(glob.glob("/kaggle/input/hms-models/final_multimodal_sanity/final_multimodal_sanity/*")):
    try:
        model = MultimodalModel(None, None, None)
        print(model_path)
        model = load_model(model_path, model)
        model = model.to(device)
        model.eval()
        models_multimodal.append(model)
    except:
        print("Corrupted:", model_path)

size = 0

# Single models
#size += len(models_eeg)
#size += len(models_spc)
#size += len(models_eeg_spc)

size += len(models_multimodal)

size

0

In [7]:
def MAD(signal, axis = -1):
    """Compute the robust standard deviation (MAD) of a signal."""
    median = np.median(signal, axis = axis, keepdims = True)
    absolute_deviations = np.abs(signal - median)
    median_absolute_deviation = np.median(absolute_deviations, axis = axis, keepdims = True)
    scale_factor = 1.4826  # This is a constant for normal distribution
    robust_std = median_absolute_deviation * scale_factor
    return robust_std

from scipy.signal import welch
from scipy.stats import linregress
from scipy.signal import butter, filtfilt

def butter_filter(eeg_data, fs = 200, cutoff_freq = 22, order = 4, btype = "lowpass"):
    b, a = butter(
        N = order, 
        Wn = cutoff_freq / (0.5 * fs), 
        btype = btype, 
        analog = False
    )

    return filtfilt(b, a, eeg_data)

In [8]:
import scipy
import torchaudio
from torchaudio.transforms import Spectrogram as _Spectrogram

n_fft = 800
#win_length = 256
win_length = 256
hop_length = 44

spectrogram = _Spectrogram(
    n_fft = n_fft, win_length = win_length, hop_length = hop_length, power = None
)

@torch.no_grad()
def compute_spec(chain):
    chain = torch.Tensor(chain)
    #chain = chain[:, :, :10000]
    chain = spectrogram(chain)
    chain = chain[:, :, 2:98]
    chain = torch.abs(chain) / 15
    chain = torch.log(chain.clip(math.exp(-4), math.exp(7)))
    chain = chain.mean(axis = 1)
    return chain.numpy()

def compute_spec_eeg(a, b) -> np.ndarray:
    return butter_filter(a - b, cutoff_freq = np.array([0.25, 40.0]), order = 5, btype = "bandpass")

def compute_spec_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy() 
    Fp2 = df_eeg["Fp2"].to_numpy()
    Fz  = df_eeg["Fz"].to_numpy()
    Cz  = df_eeg["Cz"].to_numpy()
    Pz  = df_eeg["Pz"].to_numpy()
    F3  = df_eeg["F3"].to_numpy()
    F4  = df_eeg["F4"].to_numpy()
    F7  = df_eeg["F7"].to_numpy()
    F8  = df_eeg["F8"].to_numpy()
    C3  = df_eeg["C3"].to_numpy()
    C4  = df_eeg["C4"].to_numpy()
    P3  = df_eeg["P3"].to_numpy()
    P4  = df_eeg["P4"].to_numpy()
    T3  = df_eeg["T3"].to_numpy()
    T4  = df_eeg["T4"].to_numpy()
    T5  = df_eeg["T5"].to_numpy()
    T6  = df_eeg["T6"].to_numpy()
    O1  = df_eeg["O1"].to_numpy()
    O2  = df_eeg["O2"].to_numpy()
    
    ll = np.stack([(compute_spec_eeg(Fp1 , F7), compute_spec_eeg(F7 , T3), compute_spec_eeg(T3 , T5), compute_spec_eeg(T5 , O1))])
    lp = np.stack([(compute_spec_eeg(Fp1 , F3), compute_spec_eeg(F3 , C3), compute_spec_eeg(C3 , P3), compute_spec_eeg(P3 , O1))])
    rp = np.stack([(compute_spec_eeg(Fp2 , F4), compute_spec_eeg(F4 , C4), compute_spec_eeg(C4 , P4), compute_spec_eeg(P4 , O2))])
    rl = np.stack([(compute_spec_eeg(Fp2 , F8), compute_spec_eeg(F8 , T4), compute_spec_eeg(T4 , T6), compute_spec_eeg(T6 , O2))])
    chain = np.stack([ll, lp, rp, rl])[:, 0]
    
    mads = MAD(chain, axis = -1)
    mads = np.median(mads.reshape(-1))
    chain = chain / (mads + 1e-5)
    
    chain = compute_spec(chain)
    
    return chain


def compute_spec_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_spec_chain(df_eeg)
    return chain

In [9]:
# SPECTROGRAM PROCESSING

def process_spec(spec: np.ndarray) -> np.ndarray:
    spec = spec[:, 1:]
    spec = np.stack([
        spec[:,   0:100].T,
        spec[:, 100:200].T,
        spec[:, 200:300].T,
        spec[:, 300:400].T,
    ])
    
    return spec

def compute_kaggle_spec_from_file(filepath: str) -> np.ndarray:
    spec = pl.read_parquet(filepath).to_numpy().astype(np.float32)
    spec = process_spec(spec)
    return spec

In [10]:
import numpy as np
import math

from typing import Union
from typing import Tuple
from typing import List


def bin_array(
    array, 
    bin_size, 
    axis = -1, 
    pad_dir = "symmetric", 
    mode = "edge", 
    return_padding = False,
    **padding_kwargs
) -> Union[np.ndarray, Tuple[np.ndarray, List]]:
    """Given an array and bin size, bins the array along an arbitrary axis into
    bins of size `bin_size`. It will perform padding if the array does not split
    up into equal bin sizes. 

    Args:
        array (np.ndarray): The input array.
        bin_size (int): The size of each bin.
        axis (int): The axis to bin the array along. Default is -1.
        pad_dir (str): The padding direction. One of `left`, `right`, or
            `symmetric` (default).
        return_padding (bool): Option to return the padding width used. 
        mode (str): The padding mode. See the NumPy documentation for options. 

    Returns:
        np.ndarray: The binned array where the number of bins is first. That is,
            the shape will be `(..., n_bins, bin_size, ...)`.

    """
    if axis == -1:
        axis = array.ndim - 1

    curr_len = array.shape[axis]
    n_bins = math.ceil(curr_len / bin_size)
    new_len = n_bins * bin_size

    new_shape = list(array.shape)
    new_shape[axis] = n_bins
    new_shape.insert(axis + 1, bin_size)

    # Perform padding if curr_len does not equal new_len.
    padding = [(0, 0)] * array.ndim
    if curr_len != new_len:
        if pad_dir == "left":
            pad_l = new_len - curr_len
            pad_r = 0
        elif pad_dir == "right":
            pad_l = 0
            pad_r = new_len - curr_len
        else:
            pad_l = (new_len - curr_len) // 2
            pad_r = (new_len - curr_len) - pad_l

        padding[axis] = (pad_l, pad_r)
        array = np.pad(array, padding, mode = mode, **padding_kwargs)

    array = array.reshape(new_shape)

    if return_padding:
        return array, padding

    return array

In [11]:
def compute_eeg(eeg: np.ndarray) -> np.ndarray:
    eeg = butter_filter(eeg, cutoff_freq = np.array([0.25, 50]), btype = "bandpass")
    eeg = bin_array(eeg, bin_size = 4, mode = "reflect").mean(axis = -1)
    return eeg
    
def compute_eeg_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    Fz  = df_eeg["Fz"].to_numpy()
    Cz  = df_eeg["Cz"].to_numpy()
    Pz  = df_eeg["Pz"].to_numpy()
    F3  = df_eeg["F3"].to_numpy()
    F4  = df_eeg["F4"].to_numpy()
    F7  = df_eeg["F7"].to_numpy()
    F8  = df_eeg["F8"].to_numpy()
    C3  = df_eeg["C3"].to_numpy()
    C4  = df_eeg["C4"].to_numpy()
    P3  = df_eeg["P3"].to_numpy()
    P4  = df_eeg["P4"].to_numpy()
    T3  = df_eeg["T3"].to_numpy()
    T4  = df_eeg["T4"].to_numpy()
    T5  = df_eeg["T5"].to_numpy()
    T6  = df_eeg["T6"].to_numpy()
    O1  = df_eeg["O1"].to_numpy()
    O2  = df_eeg["O2"].to_numpy()
    
    ekg = df_eeg["O2"].to_numpy()
    ekg = butter_filter(ekg, cutoff_freq = np.array([0.50, 20.0]), btype = "bandpass")
    ekg = bin_array(ekg, bin_size = 4, mode = "reflect").mean(axis = -1)
    ekg = ekg.reshape(1, -1)
    
    ll = np.stack([(compute_eeg(Fp1 - F7), compute_eeg(F7 - T3), compute_eeg(T3 - T5), compute_eeg(T5 - O1))])
    lp = np.stack([(compute_eeg(Fp1 - F3), compute_eeg(F3 - C3), compute_eeg(C3 - P3), compute_eeg(P3 - O1))])
    rp = np.stack([(compute_eeg(Fp2 - F4), compute_eeg(F4 - C4), compute_eeg(C4 - P4), compute_eeg(P4 - O2))])
    rl = np.stack([(compute_eeg(Fp2 - F8), compute_eeg(F8 - T4), compute_eeg(T4 - T6), compute_eeg(T6 - O2))])
    mid = np.stack([compute_eeg(Fz - Cz), compute_eeg(Cz - Pz)])
    
    chain = np.stack([ll, lp, rp, rl])[:, 0]
    
    return chain, mid, ekg

def compute_eeg_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    return compute_eeg_chain(df_eeg)

In [12]:
LABELS = [
    "seizure_vote", 
    "lpd_vote", 
    "gpd_vote", 
    "lrda_vote", 
    "grda_vote", 
    "other_vote"
]

In [13]:
# Option for better printing.
np.set_printoptions(formatter = {"all": lambda x: f"{x:0.3f}"})

import albumentations as A

spec_transforms = A.Compose([
    A.Resize(height = 96, width = 224)
])

def proc_kspec(x):
    x = x.copy()
        
    x = x[:, 2:98]
    x[np.isnan(x) | np.isinf(x)] = 0
    x = x.clip(np.exp(-4), np.exp(7))
    x = np.log(x)
    
    x = x - x.mean(axis = (1, 2), keepdims = True)
    x = x / (x.std(axis = (1, 2), keepdims = True) + 1e-5)
    
    x = x.transpose(1, 2, 0)
    x = spec_transforms(image = x)["image"]
    x = x.transpose(2, 0, 1)
    
    x = x.reshape(4, 96, 224)
    
    return x


def proc_eeg_spec(x):
    x = x.copy()
        
    x = x[:, :, 2:-2]
    x[np.isnan(x) | np.isinf(x)] = 0

    x = x + 1
    
    x = x.reshape(4, 96, 224)
    
    return x

def proc_eeg(eeg, mid, ekg):
    eeg = eeg.copy()
    mid = mid.copy()
    ekg = ekg.copy()

    eeg[np.isnan(eeg) | np.isinf(eeg)] = 0
    ekg[np.isnan(ekg) | np.isinf(ekg)] = 0
    mid[np.isnan(mid) | np.isinf(mid)] = 0

    eeg = eeg - eeg.mean(axis = -1, keepdims = True)
    mid = mid - mid.mean(axis = -1, keepdims = True)

    # Robust estimate of standard deviation.
    mad_std = MAD(eeg, axis = -1).reshape(-1)
    mad_std = np.median(mad_std) + 1e-5

    eeg = eeg / mad_std
    eeg = eeg.clip(-10, 10)

    mid = mid / mad_std
    mid = mid.clip(-10, 10)

    ekg = ekg / (MAD(ekg, axis = -1).reshape(-1) + 1e-5)

    eeg = eeg.reshape(16, -1)
    eeg = np.concatenate([eeg, mid, ekg], axis = 0)

    eeg = eeg.reshape(19, 2_500)

    return eeg

def flip_h(eeg):
    eeg = eeg.copy()
    eeg = eeg[::-1]
    return eeg.copy()

def flip_v(eeg):
    eeg = eeg.copy()
    eeg = eeg[:, ::-1]
    return eeg.copy()

@torch.no_grad()
def gen_ensemble_pred(df_row: pl.DataFrame) -> np.ndarray:
    eeg_id = df_row["eeg_id"].item()
    spc_id = df_row["spectrogram_id"].item()
    eeg_filepath = os.path.join(EEG_DIR, f"{eeg_id}.parquet")
    spc_filepath = os.path.join(SPEC_DIR, f"{spc_id}.parquet")

    preds = []
    
    kspec = compute_kaggle_spec_from_file(spc_filepath)
    kspec = proc_kspec(kspec)
    kspec = torch.Tensor(kspec).to(device).unsqueeze(0)
    
    eeg_spec = compute_spec_from_file(eeg_filepath)
    eeg_spec = proc_eeg_spec(eeg_spec)
    eeg_spec = torch.Tensor(eeg_spec).to(device).unsqueeze(0)
    
    eeg, mid, ekg = compute_eeg_from_file(eeg_filepath)
    eeg[np.isnan(eeg) | np.isinf(eeg)] = 0
    eeg = proc_eeg(eeg, mid, ekg)
    eeg = torch.Tensor(eeg).to(device).unsqueeze(0)

    #for model in models_eeg:
    #    model.eval()
    #    preds.append(model(eeg).exp().cpu().detach().numpy().reshape(-1))
    
    #for model in models_spc:
    #    model.eval()
    #    preds.append(model(kspec).exp().cpu().detach().numpy().reshape(-1))
    
    #for model in models_eeg_spc:
    #    model.eval()
    #    preds.append(model(eeg_spec).exp().cpu().detach().numpy().reshape(-1))
    
    for model in models_multimodal:
        model.eval()
        preds.append(model(eeg, eeg_spec, kspec).exp().cpu().detach().numpy().reshape(-1))
        
    for pred in preds:
        print(pred)
        
    preds = np.mean(preds, axis = 0)
    preds = preds / preds.sum()
    return preds

In [14]:
from tqdm.auto import tqdm
    
preds_final = []
for i in range(len(df_test)):
    pred = gen_ensemble_pred(df_test[i])
    preds_final.append(pred)

ValueError: cannot reshape array of size 141696 into shape (4,96,224)

In [15]:
df_sub = pd.DataFrame({"eeg_id": df_test["eeg_id"].to_list()})
df_sub[LABELS] = preds_final
df_sub.to_csv("submission.csv", index = False)
df_sub

ValueError: Columns must be same length as key